# Retrieval quality diagnostic — v1 vs v2

The index built fine, but the smoke-test results looked wrong. Two known issues:

1. My smoke-test cell read the wrong key (`score` instead of `similarity`),
   so every score printed as 0.000. That was a display bug, not a real one.
2. The corpus has **6,236 tafsir entries but only 1,911 unique passages** —
   Ibn Kathir comments on blocks of verses, so up to 20 verses share identical
   text. Duplicates crowd out the top-k.

This notebook measures how bad it actually is, and compares the old model (v1)
against the new one (v2) head to head.

Runtime → T4 GPU. Run all.


## 1. Setup — same as before

In [ ]:
import torch, os, shutil, subprocess
print("CUDA:", torch.cuda.is_available())

from google.colab import drive
drive.mount('/content/drive')

ROOT  = "/content/drive/MyDrive"
P1    = f"{ROOT}/Phase1_Project/MemberB_B4_B6_output"
P1FIX = f"{ROOT}/Phase1_Project/data_fix_output"
GV2   = f"{ROOT}/Phase3_Project/guardrail_output_v2"

PROJECT = "/content/QuranicRAG"
shutil.rmtree(PROJECT, ignore_errors=True)
os.makedirs(f"{PROJECT}/src", exist_ok=True)
os.makedirs(f"{PROJECT}/quranNLP/shared/data", exist_ok=True)
os.chdir(PROJECT)

subprocess.run(["git","clone","--depth","1",
  "https://github.com/Laiba-Noor/quranic-rag-hallucination-free.git","/content/_repo"],check=True)
for f in os.listdir("/content/_repo/src"):
    if f.endswith(".py"): shutil.copy(f"/content/_repo/src/{f}", f"src/{f}")
for f in os.listdir(f"{GV2}/src"):
    if f.endswith(".py"): shutil.copy(f"{GV2}/src/{f}", f"src/{f}")

shutil.copy(f"{P1FIX}/shared_data/final_cross_reference_index.csv",
            "quranNLP/shared/data/final_cross_reference_index.csv")
print("src files:", len(os.listdir("src")))

In [ ]:
!pip install -q sentence-transformers hnswlib pandas
print("ok")

## 2. Pull BOTH models and BOTH indexes down from Drive

In [ ]:
import shutil, time
t=time.time()
shutil.copytree(f"{P1}/b5_real_finetuned",      "m_v1", dirs_exist_ok=True)
shutil.copytree(f"{P1}/index",                  "i_v1", dirs_exist_ok=True)
shutil.copytree(f"{GV2}/b5_real_finetuned_v2",  "m_v2", dirs_exist_ok=True)
shutil.copytree(f"{GV2}/index_v2",              "i_v2", dirs_exist_ok=True)
print(f"copied in {time.time()-t:.0f}s")

## 3. Load both systems

In [ ]:
import sys; sys.path.insert(0,"src")
from sentence_transformers import SentenceTransformer
from b6_build_index_and_retrieval_api import load_index, RetrievalAPI

def build(mdir, idir):
    m = SentenceTransformer(mdir)
    d = m.get_sentence_embedding_dimension()
    idx, ent = load_index(dim=d, out_dir=idir)
    return RetrievalAPI(m, idx, ent), ent

api_v1, ent1 = build("m_v1", "i_v1")
api_v2, ent2 = build("m_v2", "i_v2")
print(f"v1: {len(ent1)} entries | v2: {len(ent2)} entries")

## 4. How bad is the duplicate problem?

In [ ]:
import hashlib, collections
for name, ent in [("v1", ent1), ("v2", ent2)]:
    tv = sum(1 for e in ent if e["source_type"]=="verse")
    tt = sum(1 for e in ent if e["source_type"]=="tafsir")
    uniq = len({hashlib.md5(e["text"].encode()).hexdigest() for e in ent})
    print(f"{name}: {len(ent)} entries = {tv} verses + {tt} tafsir | {uniq} unique texts "
          f"({len(ent)-uniq} duplicates, {100*(len(ent)-uniq)/len(ent):.0f}%)")

## 5. Real similarity scores — with the correct key this time

In [ ]:
QUERIES = [
    ("ما فوائد الصبر في القرآن",            "benefits of patience"),
    ("من هو النبي الذي ابتلعه الحوت",       "prophet swallowed by whale (Yunus, 37:139-148)"),
    ("قصة موسى مع فرعون",                  "Musa and Pharaoh"),
    ("ما حكم الربا في الإسلام",             "ruling on riba (2:275-279)"),
    ("قصة أصحاب الكهف",                    "People of the Cave (18:9-26)"),
]

for q, gloss in QUERIES:
    print("="*74)
    print(f"{q}    [{gloss}]")
    for name, api in [("v1", api_v1), ("v2", api_v2)]:
        rs = api.retrieve(q, top_k=5)
        top = rs[0]["similarity"]
        keys = ", ".join(f"{r['verse_key']}" for r in rs)
        print(f"  {name}  top={top:.3f}  ->  {keys}")

## 6. Same thing, but ignoring duplicate passages

In [ ]:
def retrieve_dedup(api, q, top_k=5, pool=40):
    seen, out = set(), []
    for r in api.retrieve(q, top_k=pool):
        h = r["text"][:200]
        if h in seen: continue
        seen.add(h); out.append(r)
        if len(out) >= top_k: break
    return out

for q, gloss in QUERIES:
    print("="*74)
    print(f"{q}    [{gloss}]")
    for name, api in [("v1", api_v1), ("v2", api_v2)]:
        rs = retrieve_dedup(api, q)
        print(f"  --- {name} ---")
        for r in rs[:3]:
            print(f"    [{r['verse_key']:<8}] {r['source_type']:<7} sim={r['similarity']:.3f}  {r['text'][:60]}")

## 7. The project's own diagnostic, on all 25 queries

In [ ]:
from e6_retrieval_quality_diagnostic import diagnose_retrieval_quality

test_queries = [
    "ما فوائد الصبر في القرآن", "من هو النبي المعروف بالصبر", "ماذا يقول القرآن عن الرحمة",
    "التوبة والاستغفار", "العدل في الإسلام", "الشكر لله", "الخوف من الله",
    "الايمان بالغيب", "الصدق في القول", "بر الوالدين",
    "من هو النبي المعروف بالحكمة", "ما حكم الربا في الإسلام", "أهمية الصلاة في القرآن",
    "قصة موسى مع فرعون", "من هو النبي الذي ابتلعه الحوت", "الجنة والنار في القرآن",
    "معنى التقوى", "أحكام الزكاة", "الحلال والحرام", "من هو خاتم الأنبياء",
    "قصة آدم وحواء", "الوصية بالإحسان إلى الجار", "معنى التوكل على الله",
    "قصة أصحاب الكهف", "أهمية العلم في الإسلام",
]

r1 = diagnose_retrieval_quality(api_v1, test_queries)
r2 = diagnose_retrieval_quality(api_v2, test_queries)

print(f"{'query':<34}{'v1':>8}{'v2':>8}   verdict")
print("-"*66)
better = 0
for a, b in zip(r1, r2):
    win = b.top_similarity > a.top_similarity
    better += win
    print(f"{a.query[:32]:<34}{a.top_similarity:>8.3f}{b.top_similarity:>8.3f}   {'BETTER' if win else 'worse/same'}")

import statistics as st
print("-"*66)
print(f"{'MEAN':<34}{st.mean(x.top_similarity for x in r1):>8.3f}"
      f"{st.mean(x.top_similarity for x in r2):>8.3f}")
print(f"\nv2 better on {better}/{len(r1)} queries")
print(f"low-quality (<0.5) — v1: {sum(x.is_low_quality for x in r1)}  v2: {sum(x.is_low_quality for x in r2)}")

---
## What the numbers mean

- **top similarity above ~0.6** — retrieval is working
- **0.4–0.6** — weak, the guardrail will reject a lot
- **below 0.4** — effectively random, and Phase 4 numbers would be meaningless

Send Claude the output of sections 4, 5, 6 and 7.
